In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# Define schema for sprints JSON files with StructType and StructField
sprints_schema = StructType([
    StructField("constructorId", StringType(), True),
    StructField("date", DateType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", DoubleType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("url", StringType(), True)
])

# Read sprints multiline JSON files from volume with defined schema
sprints_df = (spark.read
    .schema(sprints_schema)
    .option("mode", "FAILFAST")
    .option("multiLine", "true")  # Required for multiline JSON files
    .json(source_file)
)

#display(sprints_df)

In [0]:
# Add ingestion timestamp and source file path metadata using helper function
sprints_with_metadata_df = add_ingestion_metadata(sprints_df)

# display(sprints_with_metadata_df)

In [0]:
# Write sprints data to bronze layer table
(sprints_with_metadata_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

print(f"✅ Sprints data successfully written to {table_name} table")

In [0]:
%sql
-- SELECT 
--   constructorId,
--   date,
--   driverId,
--   grid,
--   laps,
--   number,
--   points,
--   position,
--   positionText,
--   raceName,
--   round,
--   season,
--   status,
--   url,
--   ingestion_timestamp,
--   source_file_path
-- FROM formula1.bronze.sprints
-- LIMIT 10